# Week 03 · Attention in Action — Talk Version
## 60-min session — no slides needed

**Model:** `viviktchaudhary/tiny-slm-storyteller-v1` (Qwen2.5-0.5B + QLoRA)

| Time | Act |
|------|-----|
| 0-8 min | Act 1 — Context Window |
| 8-30 min | Act 2 — Self-Attention, live |
| 30-35 min | Act 3 — Causal Mask |
| 35-50 min | Act 4 — Your SLM |
| 50-60 min | Act 5 — Play / Close |

> Session question: *"I asked the AI to recommend a restaurant near the Eiffel Tower and it made one up. Why?"*  
> We answer the *why* today. We fix it in Week 04 RAG.


## 0 · Setup

In [ ]:
# Install — run once
import sys
# !{sys.executable} -m pip install -q transformers torch matplotlib accelerate

import math, warnings
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
torch.manual_seed(42)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('\n✅ Ready')


---
## Act 1 · The Context Window
An LLM has no persistent memory. All it has is a **context window** — a box of tokens it can see.

> Everything in Week 04 RAG is about what you put in that box.


In [ ]:
def show_context_window(context_items, title="Context Window"):
    print(f"\n{'═'*58}")
    print(f"  {title}")
    print(f"{'═'*58}")
    for tag, text in context_items:
        print(f"  [{tag:<12}]  {text}")
    print(f"{'═'*58}\n")

# Scenario — bare, no grounding — this is what causes hallucination
bare_context = [
    ("SYSTEM", "You are a helpful assistant."),
    ("USER",   "Recommend a restaurant near the Eiffel Tower."),
]

show_context_window(bare_context, "Context Window (bare)")
display(Markdown("**Model response (likely):** *Le Jules Verne is a wonderful choice, located on the second floor of the Eiffel Tower…*"))
display(Markdown("**Problem:** It fills gaps with probability, not facts. This is hallucination."))
print("\n→ Week 04 RAG: we fill this window with retrieved facts first. Today: how does the model *read* what's in the window?")


---
## Act 2 · Self-Attention — live
Query → what am I looking for?  
Key → what do I offer?  
Value → what info do I carry?

Attention = softmax(QKᵀ / √d) · V


In [ ]:
def simple_attention_demo(sentence="The cat chased the mouse because it was hungry", query_word="it"):
    words = sentence.split()
    n = len(words)
    d = 16
    torch.manual_seed(42)
    
    # toy embeddings
    E = torch.randn(n, d)
    W_q = torch.randn(d, d) * 0.5
    W_k = torch.randn(d, d) * 0.5
    W_v = torch.randn(d, d) * 0.5
    
    Q = E @ W_q
    K = E @ W_k
    V = E @ W_v
    
    # find query index
    try:
        q_idx = words.index(query_word)
    except ValueError:
        q_idx = n // 2
    
    scores = Q[q_idx] @ K.T / math.sqrt(d)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    
    # plot
    fig, ax = plt.subplots(figsize=(7, 3.5))
    y_pos = np.arange(n)
    ax.barh(y_pos, weights.detach().numpy(), color='#4a7c7e')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_xlabel('Attention weight')
    ax.set_title(f'What does "{words[q_idx]}" attend to?')
    for i, w in enumerate(weights):
        ax.text(float(w)+0.01, i, f'{w:.2f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    
    print(f'Query: "{words[q_idx]}"')
    print(f'Top attend: "{words[torch.argmax(weights)]}"  weight={weights.max():.3f}')
    return weights

weights = simple_attention_demo()
print("\nTip: change query_word to 'cat', 'mouse', etc. in the cell above and re-run live.")


---
## Act 3 · Causal Mask
Why generation is left-to-right, word-by-word.


In [ ]:
def show_causal_mask(n=6):
    mask = torch.tril(torch.ones(n, n))
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(mask, cmap='Greens', vmin=0, vmax=1)
    ax.set_title('Causal mask — can only see left')
    ax.set_xlabel('Key position →')
    ax.set_ylabel('Query position ↓')
    for i in range(n):
        for j in range(n):
            ax.text(j, i, int(mask[i,j]), ha='center', va='center')
    plt.show()
    print("Green = allowed. White = masked to -inf before softmax.")
    print("This is why LLMs generate left-to-right.")

show_causal_mask()


---
## Act 4 · Your SLM — Qwen2.5-0.5B + QLoRA
`viviktchaudhary/tiny-slm-storyteller-v1`

This is Multi-Head Attention running for real. 16 heads, 24 layers, all doing what we just visualized.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "viviktchaudhary/tiny-slm-storyteller-v1"
print(f"Loading {model_id} …")
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map=None
)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("✅ SLM ready")

def generate_story(prompt="Once upon a time a brave little prince", max_new_tokens=80, temperature=0.7):
    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.cuda() if torch.cuda.is_available() else v for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tok.eos_token_id
        )
    text = tok.decode(out[0], skip_special_tokens=True)
    display(Markdown(f"**temperature={temperature}**\n\n{text}"))
    return text

# Temperature sweep — run these live
generate_story(temperature=0.7)
generate_story(temperature=1.0)
generate_story(temperature=1.3)


---
## Act 5 · Play / Close

**Audience prompt cell — type anything:**


In [ ]:
# Change this and run
my_prompt = "In a quiet village near Bangalore, a robot"
generate_story(my_prompt, max_new_tokens=100, temperature=0.8)


### Close
- Attention = softmax(QKᵀ / √d) · V — look-up over the context window
- Multi-Head = 16 of these in parallel, each catching different patterns
- Causal mask = why generation is left-to-right
- Temperature = how sharply Attention picks the next token

**Your model:** https://huggingface.co/viviktchaudhary/tiny-slm-storyteller-v1

> **Week 04 RAG cliffhanger:**  
> *"Same model, same Eiffel Tower question. Next week we fill the context window with retrieved facts first. Attention stays the same — we just give it something true to attend to."*

Links:
- Sim notebook: this file
- SLM: viviktchaudhary/tiny-slm-storyteller-v1
- Papers: Attention Is All You Need — https://arxiv.org/abs/1706.03762
